# 1. Prepare Data (2015-2019)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

## 1.1 Load Original Dataset

In [ ]:
# file_path = '/content/drive/MyDrive/Thesis/Dallas_Animal_Shelter_Data_Fiscal_Year_2000_-_Present_20260209.csv'
# Change your file path

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file at '{file_path}' was not found. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

In [ ]:
df.info()

## 1.2 Filter 2015-2019 Year Data

In [ ]:
# 1. Getting the data 2015-2019

## 1.1 making a Intake Year Column
df['Intake_Date'] = pd.to_datetime(df['Intake_Date'], errors='coerce')

df['Intake_Year'] = df['Intake_Date'].dt.year

## 1.2 Leaving data under Intake_Year bettwen 2020-2024
df_2015_2019 = df[df['Intake_Year'].between(2015,2019)].copy()
df_2015_2019['Intake_Year'].value_counts().sort_index()


In [ ]:
df_2015_2019.info()

# 180749 rows

In [ ]:
df['Reason'].value_counts()

## 1.3 Check Duplicates

* No duplicates

In [ ]:
# 2015-2019 filtering - exact duplicate
df_2015_2019.duplicated().sum()

# 2. Calculate LOS

In [ ]:
## 2.1 Check Intake/Outcome Date missing rate
print('Missing Intake Dates:', df_2015_2019['Intake_Date'].isna().sum())
print('Missing Outcome Dates:', df_2015_2019['Outcome_Date'].isna().sum())


## 2.2 Ensure datetime types
df_2015_2019['Intake_Date']  = pd.to_datetime(df_2015_2019['Intake_Date'],  errors='coerce')
df_2015_2019['Outcome_Date'] = pd.to_datetime(df_2015_2019['Outcome_Date'], errors='coerce')


## 2.3 Drop missing values from Outcome_Date
df_2015_2019 = df_2015_2019[df_2015_2019['Outcome_Date'].notna()].copy()


## 2.4 Calculate LOS
df_2015_2019['LOS_days'] = (df_2015_2019['Outcome_Date'] - df_2015_2019['Intake_Date']).dt.days


## 2.5 checking the column
print('LOS Summary: ')
print(df_2015_2019['LOS_days'].describe())
print((df_2015_2019['LOS_days'] < 0).mean()) # los < 0 does not exist



# 3. Define Analysis Population

## 3.1 Keep Cats and Dogs Only (Animal Type)

In [ ]:
df_2015_2019['Animal_Type'].value_counts()

In [ ]:
# Leave only cats and dogs from Animal_Type
df_2015_2019 = df_2015_2019[df_2015_2019["Animal_Type"].isin(["DOG", "CAT"])].copy()
df_2015_2019["LOS_days"].describe()


In [ ]:
# Check

df_2015_2019["Animal_Type"].value_counts()

## 3.2 Filter Relevant Outcome Types

In [ ]:
df_2015_2019['Outcome_Type'].value_counts()

In [ ]:
df_2015_2019.groupby('Outcome_Type')['LOS_days'].describe().sort_values(by='mean', ascending=False)

check lost/found exp

In [ ]:
cols_to_check = [
    "Outcome_Type",
    "Intake_Type",
    "Intake_Subtype",
    "Reason",
    "Animal_Type",
    "Animal_Breed",
    "Intake_Date",
    "Outcome_Date",
    "LOS_days"
]

df_2015_2019[
    df_2015_2019["Outcome_Type"].isin(["LOST EXP", "FOUND EXP"])
][cols_to_check].sample(30, random_state=42)

In [ ]:
# Clean Outcome_Type values
df_2015_2019["Outcome_Type"] = df_2015_2019["Outcome_Type"].str.strip().str.upper()

# Remove irrelevant outcome types
exclude_outcomes = ["DEAD ON ARRIVAL", "OTHER", "FOUND REPORT", "LOST EXP", "FOUND EXP"]

df_2015_2019 = df_2015_2019[
    ~df_2015_2019["Outcome_Type"].isin(exclude_outcomes)
].copy()

df_2015_2019["Outcome_Type"].value_counts()

# 4. Data Cleaning & Handling Missing Values

## 4.1 Check missing values
* Census_Tract
* Intake_Subtype
* Hold_Request
* Outcome_Condition
* Animal_Origin


In [ ]:
# Missing value Check
df_2015_2019.isna().mean()

## 4.2 Animal Related Features


### 4.2.1 Animal_Type
* Animal type distribution imbalance: approximately 3:1 dog-to-cat ratio
- Animal_Type was imbalanced, with dogs accounting for 74.4% of intake cases and cats accounting for 25.6%. This distribution was retained because it reflects the observed shelter population during the study period. Since Animal_Type was used as a predictor rather than the target variable, no resampling was applied based on animal type.

In [ ]:
# Check Animal_Types
df_2015_2019['Animal_Type'].value_counts(normalize=True)

### 4.2.2 Animal_Breed

In [ ]:
df_2015_2019["Animal_Breed_clean"] = (
    df_2015_2019["Animal_Breed"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [ ]:
# Check Animal_Breed
breed_summary = df_2015_2019['Animal_Breed_clean'].value_counts().to_frame(name='count')

breed_summary['percentage'] = (
    df_2015_2019['Animal_Breed_clean']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(breed_summary)

breed_summary['cum_percentage'] = breed_summary['percentage'].cumsum()
print(breed_summary.head(50))

In [ ]:
breed_summary['cum_percentage'] = breed_summary['percentage'].cumsum()
breed_summary.head(25)

In [ ]:
# Animal_Breed category names that contain "PIT"
pit_breed_counts = (
    df_2015_2019["Animal_Breed_clean"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .value_counts()
)

pit_breed_counts[pit_breed_counts.index.str.contains("PIT", case=False, na=False)]

In [ ]:
# unify the pit bull breeds
breed_map = {
    "AM PIT BULL TER": "PIT BULL",
    "PITBULL": "PIT BULL",
    "DOMESTIC SHORTH": "DOMESTIC SH",
    'MIXED': 'MIXED BREED',
    'CHIHUAHUA': 'CHIHUAHUA SH'
    }


df_2015_2019["Animal_Breed_clean"] = (
    df_2015_2019["Animal_Breed_clean"].replace(breed_map)
)

In [ ]:
df_2015_2019["Animal_Breed_clean"].value_counts()

## 4.3 Intake Related Features

This contains  'Intake_Type', 'Intake_Subtype', 'Intake_Condition', 'Hold_Request', 'Chip_Status', 'Animal_Origin', 'Intake_Year'

### 4.3.1 Intake Types


In [ ]:
df_2015_2019['Intake_Type'].value_counts()

### 4.3.2 Intake Subtypes
* Removed Euthanasia requested, died, urgent cases
* URGENT was excluded because it appeared only once and represented a zero-day euthanasia case, making it an extremely rare and non-representative intake pathway.

In [ ]:
df_2015_2019['Intake_Subtype'].value_counts()

In [ ]:
df_2015_2019[df_2015_2019['Intake_Subtype'] == 'URGENT'][
    ['Intake_Type', 'Intake_Subtype', 'Intake_Condition', 'Outcome_Type', 'LOS_days']
]

In [ ]:
# Basic normalization
df_2015_2019["Intake_Subtype_clean"] = (
    df_2015_2019["Intake_Subtype"]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Remove possible leakage / out-of-scope intake subtypes
exclude_subtypes = [
    "EUTHANASIA REQUESTED",
    "DIED",
    "URGENT"
]

print("Rows before:", len(df_2015_2019))
print(df_2015_2019["Intake_Subtype_clean"].value_counts().reindex(exclude_subtypes))

df_2015_2019 = df_2015_2019[
    ~df_2015_2019["Intake_Subtype_clean"].isin(exclude_subtypes)
].copy()

print("Rows after:", len(df_2015_2019))
print(df_2015_2019["Intake_Subtype_clean"].value_counts().reindex(exclude_subtypes))

In [ ]:
# missing check and impute

df_2015_2019['Intake_Subtype_clean'].isna().sum()

In [ ]:
# Remove rows with missing Intake_Subtype_clean
df_2015_2019 = df_2015_2019.dropna(subset=["Intake_Subtype_clean"]).copy()

In [ ]:
df_2015_2019['Intake_Subtype_clean'].value_counts()

In [ ]:
df_2015_2019.info()

### 4.3.3 Animal_Origin
* Missing value -> UNKNOWN

In [ ]:
df_2015_2019['Animal_Origin'].value_counts()

In [ ]:
df_2015_2019['Animal_Origin'].isna().sum()

In [ ]:
df_2015_2019['Animal_Origin'] = df_2015_2019['Animal_Origin'].fillna('UNKNOWN')

In [ ]:
df_2015_2019['Animal_Origin'].value_counts()

### 4.3.4 Chip Status

In [ ]:
df_2015_2019['Chip_Status'].value_counts()

### 4.3.5 Intake Condition

* removed dead/deceased rows

In [ ]:
df_2015_2019['Intake_Condition'].value_counts()

In [ ]:
df_2015_2019[df_2015_2019["Intake_Condition"].isin(["DEAD", "DECEASED"])][
    ["Intake_Condition", "Outcome_Type", "Intake_Subtype", "LOS_days"]
].head(30)

In [ ]:
# Exclude death-related intake condition records
df_2015_2019 = df_2015_2019[
    ~df_2015_2019["Intake_Condition"].isin(["DEAD", "DECEASED"])
].copy()

In [ ]:
df_2015_2019.info()

### 4.3.6 Intake Year

In [ ]:
df_2015_2019['Intake_Year'].value_counts()

In [ ]:
df_2015_2019.groupby('Intake_Year')['LOS_days'].describe()

In [ ]:
df_2015_2019.boxplot(column="LOS_days", by="Intake_Year", figsize=(8, 5))
plt.title("LOS Distribution by Intake Year")
plt.suptitle("")
plt.xlabel("Intake Year")
plt.ylabel("LOS days")
plt.show()

## 4.4 Geo-related Features


### 4.4.1 Census_Tract

In [ ]:
df_2015_2019['Census_Tract'].value_counts()

In [ ]:
df_2015_2019['Census_Tract'].isna().sum()

In [ ]:
def clean_census_tract(x):
    # Missing values
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    # Remove trailing .0 from float-like values
    x = x.replace(".0", "")

    # Treat invalid/unknown values
    if x in ["", "0", "UNKNOWN", "NAN", "NONE"]:
        return np.nan

    # Keep only numeric-like values
    if not x.isdigit():
        return np.nan

    # Pad to 6 digits
    return x.zfill(6)

df_2015_2019["Census_Tract_clean"] = df_2015_2019["Census_Tract"].apply(clean_census_tract)

In [ ]:
df_2015_2019[["Census_Tract", "Census_Tract_clean"]].drop_duplicates().head(30)

# 5. LOS Exploratory Analysis

## 5.1 Basic Information

* Highly Skewed (Skewness 9.89)

In [ ]:
print("LOS Info:\n", df_2015_2019['LOS_days'].describe())
print("LOS 0", (df_2015_2019['LOS_days'] == 0).mean())
print("Skewness:", df_2015_2019['LOS_days'].skew())
#

## 5.2 LOS Distribution Graph

### 5.2.1 LOS Histogram

In [ ]:
# Save directory
fig_dir = "/content/drive/MyDrive/Thesis/figures"
os.makedirs(fig_dir, exist_ok=True)

plt.figure(figsize=(8, 5))
plt.hist(df_2015_2019["LOS_days"], bins=50)
plt.title("Distribution of Length of Stay")
plt.xlabel("LOS days")
plt.ylabel("Count")
plt.tight_layout()

# Save as PNG and PDF
plt.savefig(f"{fig_dir}/los-distribution.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{fig_dir}/los-distribution.pdf", bbox_inches="tight")

plt.show()

los < 99th percentile

In [ ]:
fig_dir = "/content/drive/MyDrive/Thesis/figures"
os.makedirs(fig_dir, exist_ok=True)

los_99 = df_2015_2019["LOS_days"].quantile(0.99)

plt.figure(figsize=(8, 5))
plt.hist(
    df_2015_2019[df_2015_2019["LOS_days"] <= los_99]["LOS_days"],
    bins=50
)
plt.title("Distribution of Length of Stay (≤ 99th percentile)")
plt.xlabel("LOS days")
plt.ylabel("Count")
plt.tight_layout()

# Save BEFORE plt.show()
plt.savefig(f"{fig_dir}/los-distribution-99.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{fig_dir}/los-distribution-99.pdf", bbox_inches="tight")

plt.show()

### 5.2.2 LOG Transformation Distribution

* Raw skewness: ~9.98
* After log transformation: 0.07573052115332828
* Use log transformation in LOS regression prediction

In [ ]:
fig_dir = "/content/drive/MyDrive/Thesis/figures"
os.makedirs(fig_dir, exist_ok=True)

df_2015_2019["log_LOS_days"] = np.log1p(df_2015_2019["LOS_days"])

plt.figure(figsize=(8, 5))
plt.hist(df_2015_2019["log_LOS_days"], bins=50)
plt.title("Distribution of Log-transformed LOS")
plt.xlabel("log(LOS days + 1)")
plt.ylabel("Count")
plt.tight_layout()

# Save BEFORE plt.show()
plt.savefig(f"{fig_dir}/los-distribution-log.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{fig_dir}/los-distribution-log.pdf", bbox_inches="tight")

plt.show()

In [ ]:
df_2015_2019["log_LOS_days"].skew()

## 5.3 LOS per categorical features

In [ ]:
df_2015_2019.groupby("Animal_Type")["LOS_days"].describe().sort_values("mean", ascending=False)

In [ ]:
df_2015_2019.groupby("Intake_Type")["LOS_days"].describe().sort_values("mean", ascending=False)

In [ ]:
df_2015_2019.groupby("Intake_Condition")["LOS_days"].describe().sort_values("mean", ascending=False)

In [ ]:
df_2015_2019.groupby("Chip_Status")["LOS_days"].describe().sort_values("mean", ascending=False)

## 5.4 Due_Out - Intake_Date (Expected_LOS)

In [ ]:
df_2015_2019['Due_Out'] = pd.to_datetime(df_2015_2019['Due_Out'], errors='coerce')

In [ ]:
df_2015_2019['Expected_LOS'] = (df_2015_2019['Due_Out'] - df_2015_2019['Intake_Date']).dt.days
df_2015_2019['Error'] = df_2015_2019['LOS_days'] - df_2015_2019['Expected_LOS']

In [ ]:
df_2015_2019[['Expected_LOS', 'Error']].head(20)

In [ ]:
print(df_2015_2019[['LOS_days', 'Expected_LOS']].describe())

print(df_2015_2019[['LOS_days', 'Expected_LOS']].corr())


### EDA용으로만 남겨놓기!!!

# 6. Target Variable Construction

* Threshold 14 days
* Need to deal with class imbalance in preprocessing

In [ ]:
# Define long stay as LOS of 14 days or longer
threshold = 14

df_2015_2019["long_stay"] = (
    df_2015_2019["LOS_days"] >= threshold
).astype(int)

# Check class distribution
long_stay_summary = pd.DataFrame({
    "count": df_2015_2019["long_stay"].value_counts().sort_index(),
    "percentage": df_2015_2019["long_stay"].value_counts(normalize=True).sort_index() * 100
})

long_stay_summary.index = ["short_stay (<14 days)", "long_stay (>=14 days)"]
long_stay_summary["percentage"] = long_stay_summary["percentage"].round(2)

long_stay_summary

In [ ]:
print("Long-stay threshold:", threshold, "days")

# 7. Save Cleaned Dataframe

In [ ]:
df_clean = df_2015_2019.copy()
df_clean.to_pickle('/content/drive/MyDrive/Thesis/df_clean.pkl')

In [ ]:
df_clean.info()

In [ ]:
df_clean = pd.read_pickle('/content/drive/MyDrive/Thesis/df_clean.pkl')

df_clean.head()

In [ ]:
tract_los_summary = (
    df_clean
    .groupby("Census_Tract_clean")
    .agg(
        n=("LOS_days", "size"),
        mean_los=("LOS_days", "mean"),
        median_los=("LOS_days", "median"),
        p75_los=("LOS_days", lambda x: x.quantile(0.75)),
        max_los=("LOS_days", "max"),
        long_stay_rate=("long_stay", "mean")
    )
    .reset_index()
)

# sample size 너무 작은 tract 제외
tract_los_summary_f = tract_los_summary[tract_los_summary["n"] >= 30]

tract_los_summary_f.sort_values("median_los", ascending=False).head(20)

In [ ]:
tract_los_summary_f.sort_values("long_stay_rate", ascending=False).head(20)

In [ ]:
overall_long_stay_rate = df_clean["long_stay"].mean()
overall_median_los = df_clean["LOS_days"].median()

overall_long_stay_rate, overall_median_los

In [ ]:
tract_los_summary_f["long_stay_rate_diff"] = (
    tract_los_summary_f["long_stay_rate"] - overall_long_stay_rate
)

tract_los_summary_f["median_los_diff"] = (
    tract_los_summary_f["median_los"] - overall_median_los
)

tract_los_summary_f.sort_values("long_stay_rate_diff", ascending=False).head(20)